#### DSA 4060 – Practical Quiz: Movie Recommender
Name: Monica Njoki Wanjiku
Student ID: 670176
Assigned User ID: 37 (76 MOD 40 + 1)

In [16]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

# 1. Display first 5 rows
print("--- Movies (First 5 Rows) ---")
display(movies.head())

print("\n--- Ratings (First 5 Rows) ---")
display(ratings.head())

# 2. Show dataset dimensions (rows, columns)
print(f"Movies shape: {movies.shape} (Rows: {movies.shape[0]}, Columns: {movies.shape[1]})")
print(f"Ratings shape: {ratings.shape} (Rows: {ratings.shape[0]}, Columns: {ratings.shape[1]})")

FileNotFoundError: [Errno 2] No such file or directory: 'movies.csv'

In [ ]:
print("\nMissing values in movies.csv:")
print(movies.isnull().sum())

print("\nMissing values in ratings.csv:")
print(ratings.isnull().sum())


Missing values in movies.csv:
movie_id    0
title       0
genres      0
dtype: int64

Missing values in ratings.csv:
user_id     0
movie_id    0
rating      0
dtype: int64


In [ ]:
ASSIGNED_USER_ID = 37

# Merge user ratings with movie catalogue for complete details
user_ratings = ratings[ratings['user_id'] == ASSIGNED_USER_ID].merge(movies, on='movie_id')

# 1. List all movies rated by User 37
print(f"\n--- All Movies Rated by User {ASSIGNED_USER_ID} ---")
display(user_ratings[['movie_id', 'title', 'genres', 'rating']])

# 2. Top 3 highest-rated movies
top_3_movies = user_ratings.sort_values(by='rating', ascending=False).head(3)
print(f"\n--- User {ASSIGNED_USER_ID}'s Top 3 Highest-Rated Movies ---")
display(top_3_movies[['title', 'genres', 'rating']])


--- All Movies Rated by User 37 ---


,movie_id,title,genres,rating
0,13,Titanic,Romance|Drama,4.0
1,14,The Notebook,Romance|Drama,3.5
2,16,Crazy Rich Asians,Romance|Comedy,1.5
3,17,The Pursuit of Happyness,Drama|Biography,5.0
4,18,The Shawshank Redemption,Drama,3.5
5,19,The Social Network,Drama|Biography,4.0
6,21,Toy Story,Animation|Comedy,3.0
7,33,Ford v Ferrari,Sports|Drama,5.0
8,34,Creed,Sports|Drama,3.5
9,35,Moneyball,Sports|Drama,3.5



--- User 37's Top 3 Highest-Rated Movies ---


,title,genres,rating
3,The Pursuit of Happyness,Drama|Biography,5.0
7,Ford v Ferrari,Sports|Drama,5.0
10,Remember the Titans,Sports|Drama,5.0


In [ ]:
genre_ratings = user_ratings.assign(genres=user_ratings['genres'].str.split('|')).explode('genres')
avg_genre_ratings = genre_ratings.groupby('genres')['rating'].agg(['count', 'mean']).sort_values(by='mean', ascending=False)
print("\n--- User's Average Rating per Genre ---")
display(avg_genre_ratings)


--- User's Average Rating per Genre ---


,count,mean
genres,,
Biography,2,4.500000
Sports,4,4.250000
Drama,9,4.111111
Animation,1,3.000000
Romance,3,3.000000
Comedy,2,2.250000


In [ ]:
liked_movies = user_ratings[user_ratings['rating'] >= 4.0]
liked_movie_ids = liked_movies['movie_id'].tolist()
rated_movie_ids = user_ratings['movie_id'].tolist()

In [ ]:
movies['genres_clean'] = movies['genres'].str.replace('|', ' ')
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [ ]:
liked_indices = movies[movies['movie_id'].isin(liked_movie_ids)].index

sim_scores = cosine_sim[liked_indices].mean(axis=0)

# Attach similarity score to movies dataframe
movies_copy = movies.copy()
movies_copy['sim_score'] = sim_scores

# Exclude already rated movies
unrated_movies = movies_copy[~movies_copy['movie_id'].isin(rated_movie_ids)]

# Top 5 recommended movies
recommendations = unrated_movies.sort_values(by='sim_score', ascending=False).head(5)

In [ ]:
reasons = []
liked_genres_str = ", ".join(liked_movies['genres'].unique())

for _, row in recommendations.iterrows():
    reasons.append(f"Shares key genres ({row['genres']}) with user's top-rated movies (rated 4.0+).")

recommendations['Reason'] = reasons
recommendations['Rank'] = range(1, 6)

results_table = recommendations[['Rank', 'title', 'genres', 'Reason']].rename(
    columns={'title': 'Recommended Movie', 'genres': 'Genres'}
)

print(f"\n--- Final Top 5 Recommendations for User {ASSIGNED_USER_ID} ---")
display(results_table)


--- Final Top 5 Recommendations for User 37 ---


,Rank,Recommended Movie,Genres,Reason
19,1,Hidden Figures,Drama|Biography,Shares key genres (Drama|Biography) with user'...
29,2,The Grand Budapest Hotel,Comedy|Drama,Shares key genres (Comedy|Drama) with user's t...
3,3,Arrival,Sci-Fi|Drama,Shares key genres (Sci-Fi|Drama) with user's t...
5,4,Blade Runner 2049,Sci-Fi|Drama,Shares key genres (Sci-Fi|Drama) with user's t...
1,5,Interstellar,Sci-Fi|Drama,Shares key genres (Sci-Fi|Drama) with user's t...


Limitation: Cold Start / Novelty Bias. The content-based filter relies solely on matching exact text genre strings (e.g., Action, Drama). It ignores overall movie quality, release year, director, and popular user trends, and it will never recommend a movie outside the user's previously watched genres (serendipity problem).

Realistic Improvement: Hybrid Recommender System. Combine content-based genre filtering with Collaborative Filtering (such as Matrix Factorization / SVD or K-Nearest Neighbors). This would incorporate ratings from other similar users to discover popular, high-quality movies across diverse genres while preserving personal preferences.